In [ ]:
%load_ext autoreload
%autoreload 2

%load_ext autoreload
%autoreload 2

import torch
from argparse import Namespace

from ncpu.dataset import NCPUDataset, sample_4bit_adder
from ncpu.nca import NeuralCA
from ncpu.trainer import NCPUTrainer

import matplotlib.pyplot as plt

In [ ]:
device = "cpu"
ds_config = Namespace(
    W=117,
    H=117,
    r=6,
    spacing=(0, 58),
    sampler=sample_4bit_adder,
    balanced=False,
)

nca_config = Namespace(
    channels=16,
    hidden_channels=[100],
    fire_rate=0.9,
    alive_threshold=0.1,
    zero_initialization=True,
    mass_conserving="no",
    kernel_size=3,
    num_perception_kernels=3,
    read_only_dims=[],
)

optim_config = Namespace(
    lr=0.0001,
    batch_size=12,
    gaussian_noise=-1,
)

In [ ]:
nca_config = Namespace(
    channels=8,
    hidden_channels=[128],
    fire_rate=0.99,
    alive_threshold=0.1,
    zero_initialization=False,
    mass_conserving="no",
    kernel_size=5,
    num_perception_kernels=5,
    read_only_dims=[-1],
)

nca = NeuralCA(**vars(nca_config)).to(device)

In [ ]:
from ncpu.trainer import NCPUTrainer

optim_config = Namespace(
    lr=0.0001,
    batch_size=8,
    gaussian_noise=0.01,
)

trainer = NCPUTrainer(
    nca,
    dataset.get_dataloader(batch_size=optim_config.batch_size),
    lr=optim_config.lr,
    gaussian_noise=optim_config.gaussian_noise,
)
trainer.sanity_check()

In [ ]:
with torch.no_grad():
    info = trainer.optim_step(steps=20)
    # trainer.display_optim_step(info, display_size=117, to_show=8)
    # trainer.save_checkpoint()

In [ ]:
rollout = info["rollout"]
out = info["out"]

In [ ]:
rollout.shape, out.shape

In [ ]:
torch.unsqueeze(out, dim=1).repeat(1, 20, 1, 1).shape

In [ ]:
rollout[:, -20:, 0].shape

In [ ]:
info["nca_out"].shape

In [ ]:
screen = info["nca_out"][:, :].detach().cpu().numpy()
plt.imshow(screen[0])

In [ ]:
screen = info["nca_out"][:, :, dataset.W // 2 - 10 :].detach().cpu().numpy()
plt.imshow(screen[0])